# Feature Track 3: Retrieval Strategies

---

The baseline retriever embeds the user query and finds the nearest chunks by cosine similarity. This works well when the query is well-formed, specific, and aligned with how the data is written. In practice, this assumption often breaks.

Retrieval failures are not only about wording, they arise whenever there is a mismatch between user intent, document structure, and retrieval mechanics. Common failure modes include:
- Queries that rely on exact identifiers or codes
- Queries that mix semantic intent with keyword-level constraints
- Queries where many chunks are vaguely relevant, but only a few are actually useful
- Queries that should be restricted to a known document or subset, but aren’t

One visible manifestation of these failures is vocabulary mismatch:

> **Vocabulary mismatch** -> the right chunk exists but ranks too low because the phrasing differs:

| User query | Document text | Issue |
|---|---|---|
| `"FSC-C147829 certificate"` | `"complies with FSC standard C147829"` | Exact string -> semantic embedding is noisy |
| `"PFAS-free"` | `"no intentionally added per- and polyfluoroalkyl substances"` | Acronym vs. full name |
| `"Is this Blauer Engel certified?"` | `"Blauer Engel DE-UZ 14"` | Equivalent claims, different phrasing |


But the underlying issue is broader: single-shot semantic retrieval is a blunt tool. It optimizes for general semantic similarity, not precision, constraints, or intent.

This notebook covers three strategies that address this:

| # | Strategy | Approach | Extra cost |
|---|---|---|---|
| 1 | **Baseline** | Single semantic query | - |
| 2 | **BM25** | Keyword retrieval, no embedding | Corpus index at startup |
| 3 | **Hybrid** | Semantic + BM25 via Reciprocal Rank Fusion | Corpus index at startup |
| 4 | **Metadata filter** | Restrict semantic search to a known document | None |


Query-transformation strategies are covered in a seperate notebook

---

## Setup

**Prerequisites:** `conversational-toolkit` and `backend` must be installed in editable mode.
For **OpenAI**, set `OPENAI_API_KEY`. For **Ollama**, start `ollama serve` and pull the model.

In [ ]:
from conversational_toolkit.embeddings.openai import OpenAIEmbeddings
from conversational_toolkit.llms.base import LLMMessage, Roles, MessageContent
from conversational_toolkit.utils.retriever import build_query_with_chunks
from conversational_toolkit.vectorstores.chromadb import ChromaDBVectorStore

from sme_kt_zh_collaboration_rag.feature0_baseline_rag import (
    build_llm,
    build_vector_store,
    load_chunks,
    EMBEDDING_MODEL,
    VS_PATH,
    RETRIEVER_TOP_K,
    SYSTEM_PROMPT,
)
from sme_kt_zh_collaboration_rag.feature3_advanced_retrieval import (
    retrieve_baseline,
    retrieve_bm25,
    retrieve_hybrid,
    compare_retrieval_strategies,
    get_corpus_from_vector_store,
    print_strategy_comparison,
    retrieve_with_metadata_filter,
)

# Choose your LLM backend (only needed for the final answer cells)
BACKEND = "openai"  # "ollama", "openai", or "qwen"
FORCE_REBUILD = False  # set True to re-embed from scratch and rebuild the vector store

embedding_model = OpenAIEmbeddings(model_name=EMBEDDING_MODEL)
print(f"Embedding model: {EMBEDDING_MODEL}")

if FORCE_REBUILD or not VS_PATH.exists():
    chunks = load_chunks(max_files=None)
    chunks = [c for c in chunks if c.mime_type.startswith("text")]
    vector_store = await build_vector_store(
        chunks, embedding_model, db_path=VS_PATH, reset=FORCE_REBUILD
    )
else:
    # Vector store already exists -> open it without re-embedding
    vector_store = ChromaDBVectorStore(db_path=str(VS_PATH))
    print(
        f"Reusing existing vector store at {VS_PATH} ({vector_store.collection.count()} chunks)"
    )

llm = build_llm(backend=BACKEND)
print(f"LLM backend: {BACKEND}")

---

## Build the BM25 Corpus

**BM25** (Best Match 25) is a lexical retrieval algorithm, so it requires access to the raw text of the corpus. Unlike vector search, it cannot query an external index incrementally, the corpus must be tokenized and indexed at startup.

Indexing is necessary because BM25 precomputes:
- Term frequencies per chunk
- Document frequencies across the corpus
- Normalization statistics (e.g. average document length)

At query time, scoring is fast because these statistics are already available.

In [ ]:
print("Fetching corpus from vector store for BM25 indexing...")
corpus = await get_corpus_from_vector_store(vector_store, embedding_model, n=500)
print(f"Corpus size: {len(corpus)} chunks")
print("\nSample chunk titles:")
for c in corpus[:5]:
    src = c.metadata.get("source_file", "?")
    print(f"{src:<52} {repr(c.title)[:48]}")

---

## BM25 Retrieval

**BM25** (Best Match 25) ranks chunks using exact token matches, weighted by corpus-level statistics. There are no embeddings involved: retrieval is a pure string-based scoring operation.


BM25 combines three key ideas:
- Term Frequency Saturation: Relevance increases with term frequency, but with diminishing returns. A chunk mentioning “python” 100 times is not 10× more relevant than one mentioning it 10 times.
- Document Length Normalization: Longer chunks naturally contain more terms. BM25 corrects for this so long chunks are not unfairly favored.
- Inverse Document Frequency (IDF): Rare terms (e.g. identifiers, codes, acronyms) carry more weight than common words.

The behavior of BM25 depends on two parameters:
- k1 (typically 1.2–2.0): controls term-frequency saturation
- b (0–1): controls document-length normalization

### When BM25 has advantages over semantic search
BM25 excels when queries depend on exact lexical matches, such as:
- Identifiers, codes, or SKUs ("FSC-C147829")
- Acronyms and abbreviations ("PFAS", "SOC 2")
- Product names, standards, or regulatory labels
- Error codes or configuration keys

In these cases, semantic embeddings may blur or down-rank the most relevant chunk, while BM25 retrieves it reliably.

### When BM25 struggles
BM25 is **vocabulary-literal**. If two phrases share no tokens, BM25 treats them as unrelated:
- "carbon footprint decrease"
- "CO₂ reduction"

Semantic search handles this naturally; BM25 does not.

BM25 also struggles with:
- Paraphrases
- Synonyms
- Implicit or conceptual queries

In [ ]:
# Query where BM25 excels: exact product code / certification number
QUERY = "FSC-C147829 certificate"
KEYWORDS = ["FSC", "C147829"]
# QUERY = "Is the carbon neutrality claim for the tape product independently verified?"
# KEYWORDS = ["CO₂", "carbon", "tesa", "verified", "68%", "tesapack"]

results_bm25_exact = await retrieve_bm25(QUERY, vector_store, top_k=RETRIEVER_TOP_K)
results_semantic_exact = await retrieve_baseline(
    query=QUERY,
    embedding_model=embedding_model,
    vector_store=vector_store,
    top_k=RETRIEVER_TOP_K,
)

print(f"\nExact query: {QUERY!r}\n")
print("── BM25 " + "─" * 55)
for chunk in results_bm25_exact.chunks[:10]:
    src = chunk.metadata.get("source_file", "?")
    title = chunk.title or "(no title)"
    hit = any(kw.lower() in (src + title + chunk.content).lower() for kw in KEYWORDS)
    print(
        f"  {'✓' if hit else '·'}  score={chunk.score:.4f}  {src:<44}  {title[:38]!r}"
    )

print("\n── Semantic baseline " + "─" * 43)
for chunk in results_semantic_exact.chunks[:10]:
    src = chunk.metadata.get("source_file", "?")
    title = chunk.title or "(no title)"
    hit = any(kw.lower() in (src + title + chunk.content).lower() for kw in KEYWORDS)
    print(
        f"  {'✓' if hit else '·'}  score={chunk.score:.4f}  {src:<44}  {title[:38]!r}"
    )
print("\n✓ = chunk contains a relevant keyword; · = no match")

---

## Hybrid Retrieval: Semantic + BM25 via Reciprocal Rank Fusion

Hybrid retrieval runs both strategies in parallel and merges their ranked lists using **Reciprocal Rank Fusion (RRF)**:

```
Semantic retriever ──► ranked list A 
                                   ├── RRF merge ──► final top-k
BM25 retriever     ──► ranked list B
```

**RRF formula:**

$$\text{RRF}(d) = \sum_{r \in \text{retrievers}} \frac{1}{k + \text{rank}_r(d)}$$

where $k = 60$ (standard default from the RRF paper). Using *ranks* rather than raw scores means L2 distances and BM25 scores, which are on incomparable scales, never need to be normalised.

Key properties:
- Chunks appearing in **both** lists get the highest scores
- Chunks appearing in **only one** list still get credit
- Sub-retrievers run **in parallel** -> latency is bounded by the slower one, not their sum

In [ ]:
# Exact-term query: hybrid should match BM25's strong result
QUERY = "Are any tape products free of per- and polyfluoroalkyl substances?"
KEYWORDS = ["PFAS", "per-", "polyfluoroalkyl"]
# QUERY = "Is the carbon neutrality claim for the tape product independently verified?"
# KEYWORDS = ["CO₂", "carbon", "tesa", "verified", "68%", "tesapack"]

results_exact_hybrid = await retrieve_hybrid(
    QUERY, embedding_model, vector_store, corpus, top_k=RETRIEVER_TOP_K
)

print(f"\nQuery: {QUERY!r}\n")
results = await compare_retrieval_strategies(
    QUERY, embedding_model, vector_store, corpus, top_k=RETRIEVER_TOP_K
)
print_strategy_comparison(results, relevant_keywords=KEYWORDS, top_n=5)

---

## Metadata Filtering

All three strategies above search the **entire** vector store. When the user query is document-specific ("what are the transport emissions in the tesa EPD?"), retrieving from every document adds noise and can push the relevant chunks out of the top-k.

**Metadata filtering** restricts semantic search to a subset of documents by applying a ChromaDB `where` filter before ranking. The filter matches on metadata fields that were stored during ingestion (e.g. `source_file`, `mime_type`).

Filter syntax examples:
```python
{"source_file": {"$eq": "EPD_tesa.pdf"}}            # single file
{"$or": [{"source_file": {"$eq": "A.pdf"}},
          {"source_file": {"$eq": "B.pdf"}}]}       # two files
{"mime_type": {"$eq": "text/markdown"}}             # all markdown chunks
```

This is useful when the frontend knows which document the user is asking about (e.g. after a file selection) or when a tool passes an explicit document reference.

In [ ]:
# List available source files in the vector store
all_chunks = await vector_store.get_chunks_by_embedding(
    (await embedding_model.get_embeddings("test"))[0], top_k=200
)
source_files = sorted({c.metadata.get("source_file", "?") for c in all_chunks})
print("Source files in the vector store:")
for f in source_files:
    print(f"  {f}")

# Pick the first source file for the demo
TARGET_FILE = "EPD_cardboard_grupak_corrugated.pdf"
print(f"\nUsing filter: source_file == {TARGET_FILE!r}")

QUERY_META = "What are the environmental impacts of this product?"
KEYWORDS_META = ["CO2", "carbon", "GWP", "emission", "kg", "impact"]

results_meta = await retrieve_with_metadata_filter(
    QUERY_META,
    embedding_model,
    vector_store,
    filters={"source_file": {"$eq": TARGET_FILE}},
    top_k=RETRIEVER_TOP_K,
)
results_unfiltered = await retrieve_baseline(
    QUERY_META, embedding_model, vector_store, top_k=RETRIEVER_TOP_K
)

print(f"\nQuery: {QUERY_META!r}")
print(f"\nFiltered (source_file={TARGET_FILE!r}):")
for chunk in results_meta.chunks[:5]:
    src = chunk.metadata.get("source_file", "?")
    print(f"  {src:<50}  {(chunk.title or '')[:50]!r}")

print("\nUnfiltered (all documents):")
for chunk in results_unfiltered.chunks[:5]:
    src = chunk.metadata.get("source_file", "?")
    print(f"  {src:<50}  {(chunk.title or '')[:50]!r}")

In [ ]:
async def rag_answer_from_chunks(llm, chunks, query):
    """Generate a RAG answer from pre-retrieved chunks (no internal retrieval)."""
    prompt = build_query_with_chunks(query, list(chunks))
    messages = [
        LLMMessage(
            role=Roles.SYSTEM, content=[MessageContent(type="text", text=SYSTEM_PROMPT)]
        ),
        LLMMessage(role=Roles.USER, content=[MessageContent(type="text", text=prompt)]),
    ]
    return (await llm.generate(messages)).content


QUERY_COMPARISON = "What is the FSC-C147829 certificate?"
results = await compare_retrieval_strategies(
    query=QUERY_COMPARISON,
    embedding_model=embedding_model,
    vector_store=vector_store,
    corpus=corpus,
    top_k=RETRIEVER_TOP_K,
)

for strategy in ("baseline", "bm25", "hybrid"):
    result = results[strategy]
    print(f"\n_COMPARISON: {QUERY_COMPARISON}")
    print(f"\n{'─' * 72}")
    print(f"Strategy: {strategy.upper()}")
    print(f"{'─' * 72}")
    print("Sources used:")
    for chunk in result.chunks:
        src = chunk.metadata.get("source_file", "?")
        hit = any(
            kw.lower() in (src + (chunk.title or "") + chunk.content).lower()
            for kw in KEYWORDS
        )
        print(f"  {'✓' if hit else '·'}  {src:<50}  {repr(chunk.title or '')[:35]}")
    answer = await rag_answer_from_chunks(llm, result.chunks, QUERY_COMPARISON)
    print(f"\n{answer[0].text}")

---

## Quantitative Answer Quality Evaluation

Two reference-free RAGAS metrics, no ground truth needed:

| Metric | What it measures | Ground truth? |
|--------|------------------|---------------|
| **Faithfulness** | Fraction of answer claims directly supported by the retrieved context | No |
| **AnswerRelevancy** | Whether the answer addresses the actual question | No |

For each query, each strategy retrieves chunks and generates an answer. The RAGAS judge LLM then checks the answer against those chunks.

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings as LangChainOpenAIEmbeddings
from ragas.llms import LangchainLLMWrapper  # type: ignore[import-untyped]
from ragas.metrics import (  # type: ignore[attr-defined]
    Faithfulness as RagasFaithfulness,
    AnswerRelevancy as RagasAnswerRelevancy,
)

from conversational_toolkit.evaluation import EvaluationSample
from conversational_toolkit.evaluation.adapters import evaluate_with_ragas
from sme_kt_zh_collaboration_rag.feature1_evaluation import EVALUATION_QUERIES

In [ ]:
ragas_llm = LangchainLLMWrapper(
    ChatOpenAI(model="gpt-4o-mini", max_completion_tokens=8192)
)
ragas_embeddings = LangChainOpenAIEmbeddings(model="text-embedding-3-small")

queries = [q["query"] for q in EVALUATION_QUERIES]

# For each strategy, retrieve chunks, generate an answer, and build an EvaluationSample.
strategy_samples: dict[str, list[EvaluationSample]] = {
    s: [] for s in ("baseline", "bm25", "hybrid")
}

for query in queries:
    results_q = await compare_retrieval_strategies(
        query, embedding_model, vector_store, corpus, top_k=RETRIEVER_TOP_K
    )
    for strategy in ("baseline", "bm25", "hybrid"):
        chunks = list(results_q[strategy].chunks)
        answer = await rag_answer_from_chunks(llm, chunks, query)
        strategy_samples[strategy].append(
            EvaluationSample(
                query=query,
                answer=answer[0].text if answer else "",
                retrieved_chunks=chunks,
            )
        )

print(f"Built {len(queries)} samples x 3 strategies ({len(queries) * 3} total)")

In [ ]:
metrics = [
    RagasFaithfulness(),  # type: ignore[call-arg]
    RagasAnswerRelevancy(strictness=1),  # type: ignore[call-arg]
]

reports = {}
for strategy, samples in strategy_samples.items():
    print(f"Evaluating {strategy}...")
    reports[strategy] = evaluate_with_ragas(
        samples=samples,
        metrics=metrics,
        llm=ragas_llm,
        embeddings=ragas_embeddings,
    )

metric_names = list(next(iter(reports.values())).summary().keys())
print(f"\n{'Strategy':<12}  " + "  ".join(f"{m:>20}" for m in metric_names))
print("─" * (14 + 22 * len(metric_names)))
for strategy, report in reports.items():
    print(
        f"{strategy:<12}  "
        + "  ".join(f"{s:>20.3f}" for s in report.summary().values())
    )

---

## Summary

| Strategy | Best for | Limitation |
|---|---|---|
| **Baseline (semantic)** | Vocabulary mismatch, paraphrases | Fails on exact terms, IDs |
| **BM25** | Exact terms, product codes, acronyms | Fails on semantic queries |
| **Hybrid** | Both - consistent across query types | BM25 corpus must fit in memory |
| **Metadata filter** | Document-specific queries (known source) | Requires the caller to know the target document |